In [1]:
from math import pi
from ngsolve import *
import numpy as np
import scipy.sparse as sp
from netgen.geom2d import SplineGeometry
from ngsolve.meshes import MakeStructured3DMesh

from ngsolve.solvers import Newton
from ngsolve.webgui import Draw
import warnings
warnings.filterwarnings('ignore')

In [2]:
def logDensity(
nx = 20,
m = 2,
s0 = 3,
order = 1,
dt0 = 1e-2,
t = Parameter(0),
tstop = 4,
plot = 0,
cutoff = 1e-14,
quads = 0
):
    cutOff = 1e-14
    order = 1
    eps0 = 0 
    dim = 3
    dt = Parameter(dt0)
    
    k = 1/(m-1+2/dim)   
    xmin = -6
    xmax = 6
    tend = tstop
    t0 = 1
    # reference solution
    val = 1-k*(m-1)/2/dim/m*((x**2+y**2+z**2)/(t+t0)**(2*k/dim)) 
    rhoex = (t+t0)**(-k)*(IfPos(val, val**(1/(m-1)), 0)+eps0)
    uex = log(rhoex)
    

    if quads:
        mesh = MakeStructured3DMesh(hexes=quads, nx=nx, ny=nx, nz=nx,
                                            mapping=lambda x,y,z:(xmin+(xmax-xmin)*x, 
                                                                 xmin+(xmax-xmin)*y,
                                                                 xmin+(xmax-xmin)*z))
        ELE = HEX
        ir = IntegrationRule(points = [(0,0,0), 
                                       (1,0,0), 
                                       (0,1,0),
                                       (1,1,0),
                                       (0,0,1), 
                                       (1,0,1), 
                                       (0,1,1),
                                       (1,1,1),                                               
                                      ], weights = [1/8, 1/8, 1/8, 1/8,
                                                   1/8, 1/8, 1/8, 1/8] )
        et = "hex"
    else: # unstructured trig mesh
        geo = CSGeometry()
        sphere = Sphere(Pnt(0,0,0),xmax)
        geo.Add(sphere)
        mesh = Mesh(geo.GenerateMesh(maxh=12/nx))
        ELE = TET
        ir = IntegrationRule(points = [(0,0,0), (1,0,0), (0,1,0),(0,0,1)], 
                             weights = [1/24, 1/24, 1/24, 1/24] )
        et = "tet"
        
    print(mesh.ne)
    V = H1(mesh,order=order)
    fes = V
    uh = GridFunction(fes)
    
    # Used to store data of previous iteration
    uh0 = GridFunction(V)
    rho0 = GridFunction(V)
    # Used in Newton's iterations
    rhok = GridFunction(V)
    rhouk = GridFunction(V)

    rho0.vec[:] = 0
    rhok.vec[:] = 0

    u = fes.TrialFunction()
    v = fes.TestFunction() 


    a = BilinearForm(fes)
    a += m*rho0**m*grad(u)*grad(v)*dx(intrules={ELE:ir})
    a += rhok*u*v/dt*dx(intrules={ELE:ir})

    f = LinearForm(fes)
    f += (rhouk-rhok+rho0)*v/dt*dx(intrules={ELE:ir})

    # initial data: point interpolation
    uh0.Set(uex)

    # Avoid NAN values
    pos = uh0.vec.FV().NumPy() > -40
    uh0.vec.FV().NumPy()[~pos] = -np.inf
    rho0.vec.FV().NumPy()[:] = np.exp(uh0.vec.FV().NumPy())

    activeDofs = BitArray(fes.ndof)


    uList = []
    def myNewton(damp0=1, tol0 = 1e-8):
        uh.vec.data = uh0.vec
        rhok.vec.FV().NumPy()[:] = np.exp(uh.vec.FV().NumPy()[:])
        # set -inf to zero for simplicity!!! FIXME LATER
        pos = rhok.vec.FV().NumPy()==0
        uh.vec.FV().NumPy()[pos] = 0 
        rhouk.vec.FV().NumPy()[:] = rhok.vec.FV().NumPy()[:]*uh.vec.FV().NumPy()[:]

        count = 0
        ener0 = (uh.vec.FV().NumPy()[:]-1).dot(rhok.vec.FV().NumPy()[:])

        tol = abs(ener0)*tol0 # relative tolerance
        while True:
            count += 1
            # hack zero
            a.Assemble()
            f.Assemble()
            if count==1: # locate active Dofs
                rows,cols,vals = a.mat.COO()
                A = sp.csr_matrix((vals,(rows,cols)))
                active = A.diagonal()>cutOff
                activeDofs[:] = 0
                # THIS IS SUPER SLOW...
                for i in range(len(active)):
                    if active[i]:
                        activeDofs[i] = 1



            uh.vec.data = a.mat.Inverse(freedofs = activeDofs,inverse="sparsecholesky")*f.vec
            rhok.vec.FV().NumPy()[active] = np.exp(uh.vec.FV().NumPy()[active])
            rhouk.vec.FV().NumPy()[active] = rhok.vec.FV().NumPy()[active]*uh.vec.FV().NumPy()[active]


            ener1 = (uh.vec.FV().NumPy()[active]-1).dot(rhok.vec.FV().NumPy()[active])
            err = abs(ener1-ener0)
            ener0 = ener1
            if err < tol:
                uh.vec.FV().NumPy()[~active] = -np.inf
                break
            if np.isnan(err) or count==30:
                print(count,"FAILED")
                stop 

        return count 


    pnts_x = np.linspace(xmin,xmax,nx+1)  
    
    step = 0

    while t.Get() < tend-dt.Get()/2:
        t.Set(t.Get()+dt.Get())
        step += 1
        
        ct = myNewton()
        uh0.vec.data = uh.vec
        rho0.vec.FV().NumPy()[:] = np.exp(uh0.vec.FV().NumPy())
        
            
    if plot==True:                
        scene = Draw(rhok, mesh)
    error = sqrt((rhok-rhoex)**2)
    r0 = (x**2+y**2+z**2)**0.5
    sub = IfPos(r0-3,0,1)
    errL2 = sqrt(Integrate(sub*(rhok-rhoex)**2, mesh))
    
    vtk = VTKOutput(ma=mesh,
                coefs=[rhok,error],
                names = ["density",'error'],
                filename="logDensity3DBarenblatt_m_{}_elements_{}".format(m,mesh.ne)+et,
                subdivision=3)
    vtk.Do()
    
    return errL2

In [28]:
m=4
nx = 17
dt = 1e-2
logDensity(m=4,dt0=dt,t = Parameter(0),tstop = .2,nx = nx, plot=1,quads=0)

9216


WebGuiWidget(value={'ngsolve_version': '6.2.2105-211-g63bbcb022', 'mesh_dim': 3, 'order2d': 2, 'order3d': 2, '…

0.10187361168206709

In [3]:
m=4
nx = 21
dt = 1e-2
logDensity(m=4,dt0=dt,t = Parameter(0),tstop = .2,nx = nx, plot=1,quads=1)

9261


WebGuiWidget(value={'ngsolve_version': '6.2.2105-211-g63bbcb022', 'mesh_dim': 3, 'order2d': 2, 'order3d': 2, '…

0.04913092434290177